# Load X and y

In [28]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

import joblib
import json
import os

# Load features and target
X = pd.read_csv("../data_processed/X_features.csv")
y = pd.read_csv("../data_processed/y_target.csv")["efficiency_score"]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Convert timestamp to datetime
X["timestamp"] = pd.to_datetime(X["timestamp"], errors="coerce")

# Create time features
X["hour"] = X["timestamp"].dt.hour
X["weekday"] = X["timestamp"].dt.weekday    # 0=Mon, 6=Sun
X["month"] = X["timestamp"].dt.month

# Drop raw timestamp if you don't want to use it directly
X = X.drop(columns=["timestamp"])

print("After adding time features:")
print(X.columns)
display(X.head(3))


display(X.head(3))
display(y.head(3))

print("Target value counts:")
print(y.value_counts().sort_index())


X shape: (548411, 6)
y shape: (548411,)
After adding time features:
Index(['type', 'district', 'province', 'lat', 'lon', 'hour', 'weekday',
       'month'],
      dtype='object')


,type,district,province,lat,lon,hour,weekday,month
0,{ความสะอาด},บางซื่อ,กรุงเทพมหานคร,13.81865,100.53084,19,4,9
1,"{น้ำท่วม,ร้องเรียน}",ประเวศ,กรุงเทพมหานคร,13.67891,100.66709,21,6,9
2,{น้ำท่วม},บางซื่อ,กรุงเทพมหานคร,13.81853,100.53099,17,3,10


,type,district,province,lat,lon,hour,weekday,month
0,{ความสะอาด},บางซื่อ,กรุงเทพมหานคร,13.81865,100.53084,19,4,9
1,"{น้ำท่วม,ร้องเรียน}",ประเวศ,กรุงเทพมหานคร,13.67891,100.66709,21,6,9
2,{น้ำท่วม},บางซื่อ,กรุงเทพมหานคร,13.81853,100.53099,17,3,10


0    1
1    2
2    1
Name: efficiency_score, dtype: int64

Target value counts:
efficiency_score
0    120918
1     93441
2     88763
3     75811
4    105627
5     63851
Name: count, dtype: int64


# Train/test split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)


X_train: (438728, 8) X_test: (109683, 8)
y_train: (438728,) y_test: (109683,)


# Define categorical & numeric columns

In [ ]:
categorical_features = ["type", "district", "province"]
numeric_features = [ "lat", "lon"]


# Preprocessing pipeline

In [31]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


# Define models

In [32]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        multi_class="auto",
        n_jobs=-1
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=15,
        min_samples_leaf=50,
        random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingClassifier(
        random_state=42
    ),
    "knn": KNeighborsClassifier(
        n_neighbors=15
    ),
}


# Helper to train & evaluate each model

In [33]:
def eval_model(name, model, X_train, y_train, X_test, y_test):
    """
    Fits the model (with preprocessing) and prints metrics.
    Returns (fitted_pipeline, metrics_dict).
    """
    # Build pipeline: preprocess -> model
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model),
        ]
    )

    print(f"\n====================")
    print(f"Training model: {name}")
    print("====================")
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1-score: {f1_macro:.3f}")

    metrics = {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
    }

    return pipe, metrics


# Train all models and collect metrics

In [34]:
fitted_models = {}
all_metrics = {}

for name, clf in models.items():
    pipe, metrics = eval_model(name, clf, X_train, y_train, X_test, y_test)
    fitted_models[name] = pipe
    all_metrics[name] = metrics

print("\nSummary of metrics for all models:")
for name, m in all_metrics.items():
    print(
        f"{name:20s} | "
        f"Accuracy: {m['accuracy']:.3f} | "
        f"Macro F1: {m['f1_macro']:.3f}"
    )



Training model: logistic_regression


ValueError: A given column is not a column of the dataframe

# Choose the best model

In [ ]:
# Choose model with highest macro F1
best_name = max(all_metrics, key=lambda k: all_metrics[k]["f1_macro"])
best_model = fitted_models[best_name]
best_metrics = all_metrics[best_name]

print("\nBest model:", best_name)
print("Best metrics:", best_metrics)



Best model: logistic_regression
Best metrics: {'accuracy': 0.40947092986151, 'f1_macro': 0.3552282690324104}


# Save best model + metrics

In [ ]:
os.makedirs("../ML/models", exist_ok=True)
os.makedirs("../ML/results", exist_ok=True)

# Save best model
model_path = f"../ML/models/best_efficiency_model_{best_name}.pkl"
joblib.dump(best_model, model_path)

# Save all metrics
metrics_path = "../ML/results/metrics_5_models.json"
with open(metrics_path, "w") as f:
    json.dump(
        {
            "best_model": best_name,
            "best_metrics": best_metrics,
            "all_metrics": all_metrics,
        },
        f,
        indent=2,
        ensure_ascii=False  # keep Thai readable
    )

print("\nSaved best model to:", model_path)
print("Saved metrics to:", metrics_path)



Saved best model to: ../ML/models/best_efficiency_model_logistic_regression.pkl
Saved metrics to: ../ML/results/metrics_5_models.json
